# 책이지 × 공개 모델 2종: 코랩에서 설치 없이 실측

이 노트북 하나로 다음을 **자동으로** 합니다(T4 GPU로 10~15분, GPU 없이 돌리면 더 걸림 — 추정).

1. 책이지 저장소(공개)를 받습니다.
2. 공개 모델 실행기(Ollama)를 코랩 안에 설치하고 켭니다.
3. **엑사원 3.5(2.4B)** 와 **라마 3.2(3B)** 를 받습니다.
4. 두 모델에 책이지 서버(파이썬 참조 서버)를 붙여 **같은 지시문·같은 문의 5건**을 처리합니다(도구는 상담 묶음 하나).
5. 실측 기록표(`model2_record_colab_….md`)를 만들어 내려받습니다.

**쓰는 법** — 메뉴 「런타임 → 모두 실행」을 누릅니다.

- 「이 노트북은 Google에서 작성하지 않았습니다」 창이 뜨면 「무시하고 계속」을 누릅니다(공개 저장소의 노트북이라 뜨는 안내).
- 이 노트북은 T4 GPU로 열리게 해 두었습니다. GPU를 쓸 수 없다는 창이 뜨면 「GPU 없이 연결」을 고릅니다(더 오래 걸림).
- ③ 모델 받기는 몇 분 동안 글이 안 바뀌어도 정상입니다.
- 다 끝나면 기록표 파일이 내려받아지고, 같은 내용이 맨 아래 칸에 표로 보입니다.

**알아 둘 것** — 인증키는 필요 없습니다(책이지 중계로 오늘 자료). 대출 상태는 정보나루 조회일 **전일 기준**이고, 접수는 하지 않습니다. 기록표는 이 노트북이 실제로 돌린 결과만 담으며, 실패도 그대로 적습니다.

In [ ]:
# ① 저장소 받기
import os, subprocess, sys, time, json, datetime, pathlib, shutil, urllib.request
KST = datetime.timezone(datetime.timedelta(hours=9))
os.chdir("/content")
if not pathlib.Path("chaekiji").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/altarry-prog/chaekiji.git"], check=True)
else:
    subprocess.run(["git", "-C", "chaekiji", "pull", "-q"], check=True)
os.chdir("/content/chaekiji/server")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pyyaml"], check=True)
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("저장소 준비 완료 · 커밋", COMMIT)

In [ ]:
# ② 공개 모델 실행기(Ollama) 설치·시작 — 코랩 안에서만 돈다(내 PC에 아무것도 깔리지 않음)
if not shutil.which("ollama"):
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
OLLAMA = "http://127.0.0.1:11434"
_log = open("/content/ollama_serve.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=_log, stderr=_log, env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434"))
for _ in range(90):
    try:
        urllib.request.urlopen(OLLAMA + "/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
OLLAMA_VER = json.load(urllib.request.urlopen(OLLAMA + "/api/version"))["version"]
GPU = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip() or "없음(CPU)"
print("Ollama", OLLAMA_VER, "· GPU", GPU)

In [ ]:
# ③ 모델 받기 — 엑사원 3.5 2.4B(LG, 한국어) · 라마 3.2 3B(메타, 도구 호출 공식 지원)
MODELS = ["exaone3.5:2.4b", "llama3.2:3b"]
for m in MODELS:
    t0 = time.time()
    print(f"{m} 받는 중… (1.6~2GB, 몇 분 걸릴 수 있음)", flush=True)
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"{m} 받기 완료 ({time.time() - t0:.0f}초)", flush=True)

In [ ]:
# ④ 같은 문의 5건을 두 모델로 — 같은 서버(server.py)·같은 지시문(프롬프트 consult)·도구는 상담 묶음 하나(consult_request)
#    (10/18 첫 실측: 작은 모델에 도구 11종을 다 보이면 찾기 도구만 부르고 추측으로 답했다. 한 문의가 멈춰도 다음 문의로 넘어간다)
ASKS = [
    ("나미야 잡화점의 기적 빌리고 싶어요, 상동에서 받을게요", "상동", "ko"),
    ("소년이 온다 있나요? 꿈빛도서관에서 받을게요", "꿈빛", "ko"),
    ("채식주의자 상호대차 되나요?", "상동", "ko"),
    ("불편한 편의점 원미도서관으로 가져다줄 수 있어요?", "원미", "ko"),
    ("Tôi muốn mượn cuốn 나미야 잡화점의 기적", "상동", "vi"),
]
OUT_DIR = pathlib.Path("lab/out"); OUT_DIR.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(KST).strftime("%Y-%m-%d_%H%M")
CALLS = OUT_DIR / f"host_log_colab_{STAMP}.ndjson"
ANSWERS = OUT_DIR / f"host_answers_colab_{STAMP}.ndjson"
for m in MODELS:
    for ask, pickup, lang in ASKS:
        t0 = time.time()
        try:
            r = subprocess.run([sys.executable, "hosts/ollama_host.py", "--model", m, "--ollama", OLLAMA, "--mode", "auto",
                                "--pickup", pickup, "--lang", lang, "--ask", ask, "--log", str(CALLS), "--answers", str(ANSWERS),
                                "--tools", "consult", "--timeout", "900"], capture_output=True, text=True, encoding="utf-8", timeout=1800)
            out = (r.stdout or "")[-1800:] or (r.stderr or "")[-800:]
        except subprocess.TimeoutExpired:
            out = "(시간 초과 30분 — 이 문의는 답 없음으로 기록하고 다음 문의로 넘어갑니다)"
        print(f"\n━━ {m} · 「{ask}」 ({time.time() - t0:.0f}초)")
        print(out)

In [ ]:
# ⑤ 실측 기록표 만들기 → 내려받기
def _rows(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")] if pathlib.Path(p).exists() else []
calls, answers = _rows(CALLS), _rows(ANSWERS)
KEY = {"visit": "바로 이용", "ill": "관내 상호대차", "reserve": "예약", "sea": "책바다", "unknown": "미확정"}
def summary(model, ask):
    cs = [c for c in calls if c["model"] == model and c.get("ask") == ask]
    an = next((a for a in answers if a["model"] == model and a["ask"] == ask), None)
    consult = [c for c in cs if c["tool"] == "consult_request" and not c["error"]]
    route = next((c.get("route") for c in reversed(cs) if c.get("route")), None)
    text = (an or {}).get("answer", "")
    ok = bool(consult) and bool(text.strip())
    return {"model": model, "ask": ask, "tools": " → ".join(c["tool"] for c in cs) or "(도구 호출 없음)",
            "route": route, "route_label": next((c.get("route_label") for c in reversed(cs) if c.get("route_label")), "—"),
            "asof": next((c.get("asof") for c in reversed(cs) if c.get("asof")), "—"),
            "data_mode": next((c.get("mode") for c in reversed(cs) if c.get("mode")), "—"),
            "protocol": (an or {}).get("protocol") or (cs[-1]["protocol"] if cs else "—"),
            "seconds": round((an or {}).get("total_ms", 0) / 1000, 1), "ok": ok,
            "relayed": bool(route) and KEY.get(route, "§") in text, "answer": text,
            "no_tool": (not consult) and bool(text.strip()) and not text.startswith("(멈춤)")}
S = [summary(m, a[0]) for m in MODELS for a in ASKS]
now = datetime.datetime.now(KST).strftime("%Y-%m-%d %H:%M")
L = [f"# 공개 모델 2종 구동 실측 기록 — 코랩 ({now} KST)", "",
     f"- 실행 환경: Google Colab · GPU {GPU} · Ollama {OLLAMA_VER}",
     f"- 책이지: 저장소 커밋 `{COMMIT}` · 같은 서버(`server.py`) · 같은 지시문(프롬프트 `consult`) · 같은 문의 {len(ASKS)}건 · 모델에 보이는 도구는 상담 묶음 `consult_request` 하나",
     "- 원 기록: " + f"`{CALLS.name}`(도구 호출) · `{ANSWERS.name}`(최종 답) — 이 노트북이 실제로 돌린 결과이며 사람이 고치지 않았다", "",
     "## 한눈에", "", "| 모델 | 방식 | 성공 | 판정을 답에 옮김 | 도구 없이 답함 | 평균 시간 |", "|---|---|---|---|---|---|"]
for m in MODELS:
    ss = [s for s in S if s["model"] == m]
    proto = ", ".join(sorted({s["protocol"] for s in ss}))
    avg = sum(s["seconds"] for s in ss) / max(len(ss), 1)
    L.append(f"| {m} | {proto} | {sum(s['ok'] for s in ss)}/{len(ss)} | {sum(s['relayed'] for s in ss)}/{len(ss)} | {sum(s['no_tool'] for s in ss)}/{len(ss)} | {avg:.1f}초 |")
same = sum(1 for a in ASKS if len({s["route"] for s in S if s["ask"] == a[0]}) == 1 and all(s["route"] for s in S if s["ask"] == a[0]))
L += ["", f"두 모델의 판정이 같은 문의: **{same}/{len(ASKS)}건** (판정은 도구가 정하므로 같아야 정상)", "",
      "성공 = 상담 도구가 오류 없이 불렸고 최종 답이 있음 · 판정을 답에 옮김 = 도구가 정한 갈래 이름이 모델의 답에 들어 있음", "",
      "도구 없이 답함 = 상담 도구의 결과 없이 모델이 스스로 쓴 답. 판정 근거가 없고 지어낸 내용이 섞일 수 있어 창구에서 쓰면 안 된다 · (멈춤) = 같은 오류가 되풀이되거나 호출 한도에 닿아 호스트가 멈춘 문의", ""]
for i, (ask, pickup, lang) in enumerate(ASKS, 1):
    L += [f"## 문의 {i}. {ask}", "", f"받을 도서관 {pickup} · 언어 {lang}", "",
          "| 모델 | 부른 도구 | 판정 | 자료 | 시간 |", "|---|---|---|---|---|"]
    for s in [s for s in S if s["ask"] == ask]:
        L.append(f"| {s['model']} | {s['tools']} | {s['route_label']} | {s['data_mode']} · {s['asof']} | {s['seconds']}초 |")
    for s in [s for s in S if s["ask"] == ask]:
        body = s["answer"].strip().replace("\n", "<br>") or "(답 없음)"
        L += ["", f"<details><summary>{s['model']}의 답 원문</summary>", "", body, "", "</details>"]
    L.append("")
L += ["---", "대출 상태는 정보나루 조회일 전일 기준이며, 이 기록의 어떤 상담도 실제 접수로 이어지지 않았다."]
REC = OUT_DIR / f"model2_record_colab_{STAMP}.md"
REC.write_text("\n".join(L), encoding="utf-8")
try:                                   # 코랩에서는 기록표 전체를 표로 보인다
    from IPython.display import Markdown, display
    display(Markdown("\n".join(L)))
except ImportError:
    print("\n".join(L[:14]))
try:
    from google.colab import files
    files.download(str(REC))
    print("\n내려받기를 시작했습니다:", REC.name, "— 이 파일이 실측 기록입니다")
except ImportError:
    print("\n기록 파일:", REC)

## 다 끝나면

- 내려받은 `model2_record_colab_….md`가 실측 기록표입니다. 두 모델이 같은 도구(`consult_request`)를 불렀는지, 판정 갈래가 같은지를 보면 됩니다. 담당자는 이 파일을 저장소 `server/lab/out/`에 올립니다.
- 실패한 문의가 있어도 그대로 두세요. 실패도 기록입니다(구동안내서의 「알려진 제약」).
- 코랩을 닫으면 설치한 것은 모두 사라집니다. 내 PC에는 아무것도 남지 않습니다.